In [5]:
# Install packages
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd

In [7]:
data = pd.read_csv("../data/no_missing_animal_id.csv")

data.head()

,animal_id,lactation_number,days_in_milk,reproduction_status,event_date,avg_milk_flow,flow_30_60_session,yield_first_2min_session,yield_session,duration_session_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
3,5.803650e+18,1.0,394.0,Open,2019-12-14,3.492661,4.399846,8.173735,11.793402,198,1
4,-1.713215e+18,5.0,288.0,Pregnant,2020-05-11,3.311224,4.300056,7.466130,8.572896,153,2


## Sort records from earliest to latest

In [8]:
# Load the cleaned file
no_missing_animal_id = pd.read_csv("../data/no_missing_animal_id.csv")

# Sort records from earliest to latest
no_missing_animal_id["event_date"] = pd.to_datetime(no_missing_animal_id["event_date"])
no_missing_animal_id = no_missing_animal_id.sort_values("event_date").reset_index(drop=True)

# Check earliest and latest records
print("Earliest records:")
print(no_missing_animal_id[["animal_id", "event_date"]].head())

print("Latest records:")
print(no_missing_animal_id[["animal_id", "event_date"]].tail())


Earliest records:
      animal_id event_date
0 -8.344568e+18 2019-06-14
1  7.065605e+18 2019-06-14
2  3.772440e+18 2019-06-14
3 -2.738539e+18 2019-06-14
4 -8.573159e+18 2019-06-14
Latest records:
            animal_id event_date
6650765  2.913070e+18 2021-10-24
6650766 -8.610394e+18 2021-10-24
6650767  7.482694e+18 2021-10-24
6650768 -1.177337e+18 2021-10-24
6650769 -9.898323e+17 2021-10-24


## Splitting sets

In [9]:
# Sort known-ID records from earliest to latest
sorted_data = (
    no_missing_animal_id
    .sort_values("event_date")
    .reset_index(drop=True)
)

# Calculate split points
n = len(sorted_data)
train_end = int(n * 0.70)
validation_end = int(n * 0.80)

# Split the records
train = sorted_data.iloc[:train_end].copy()
validation = sorted_data.iloc[train_end:validation_end].copy()
test = sorted_data.iloc[validation_end:].copy()

# Check sizes and date ranges
for name, subset in [
    ("Training", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(
        name,
        "rows:", len(subset),
        "dates:", subset["event_date"].min(),
        "to", subset["event_date"].max()
    )

# Check the professor's requirement
train_ids = set(train["animal_id"])

print(
    "Validation rows with IDs absent from training:",
    (~validation["animal_id"].isin(train_ids)).sum()
)
print(
    "Test rows with IDs absent from training:",
    (~test["animal_id"].isin(train_ids)).sum()
)

Training rows: 4655539 dates: 2019-06-14 00:00:00 to 2021-03-01 00:00:00
Validation rows: 665077 dates: 2021-03-01 00:00:00 to 2021-05-21 00:00:00
Test rows: 1330154 dates: 2021-05-21 00:00:00 to 2021-10-24 00:00:00
Validation rows with IDs absent from training: 28045
Test rows with IDs absent from training: 232434


## Modeling

In [10]:
import pandas as pd

# Define the target and measurement features
target = "animal_id"

measurement_features = [
    "flow_30_60_session",
    "yield_first_2min_session",
    "yield_session",
    "duration_session_sec",
    "milking",
    "event_date",
]

# Convert event_date into numeric calendar features
def build_features(records):
    X = records[measurement_features].copy()
    dates = pd.to_datetime(records["event_date"], errors="raise")

    X["event_month"] = dates.dt.month
    X["event_day_of_year"] = dates.dt.dayofyear

    return X

X_train = build_features(train)
X_valid = build_features(validation)
X_test = build_features(test)

# Treat each animal ID as a category
y_train = train[target].astype("string")
y_valid = validation[target].astype("string")
y_test = test[target].astype("string")

n_cows = y_train.nunique()
n_data_points = len(X_train)


In [11]:
print("Model candidate: Random Forest classifier")
print("Number of training cows:", n_cows)
print("Number of training data points:", n_data_points)
print("Number of features:", X_train.shape[1])
print("Milking column type:", X_train["milking"].dtype)

Model candidate: Random Forest classifier
Number of training cows: 7920
Number of training data points: 4655539
Number of features: 8
Milking column type: int64


In [12]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Define target and feature columns
target = "animal_id"

feature_cols = [
    "flow_30_60_session",
    "yield_first_2min_session",
    "yield_session",
    "duration_session_sec",
    "milking",
    "event_date"
]

def build_features(records):
    X = records[feature_cols].copy()

    # Convert date to numeric features
    dates = pd.to_datetime(X["event_date"], errors="raise")

    X["event_month"] = dates.dt.month
    X["event_day_of_year"] = dates.dt.dayofyear
    X["event_weekday"] = dates.dt.dayofweek
    X["event_year"] = dates.dt.year

    # Drop raw date column
    X = X.drop(columns=["event_date"])

    # Keep milking as string to avoid dtype issues
    X["milking"] = X["milking"].astype(str).str.strip()

    return X

X_train = build_features(train)
X_valid = build_features(validation)
X_test = build_features(test)

y_train = train[target].astype("string")
y_valid = validation[target].astype("string")
y_test = test[target].astype("string")

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median"))]),
            numeric_cols
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_cols
        )
    ]
)

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", RandomForestClassifier(
        n_estimators=30,
        max_depth=12,
        min_samples_leaf=5,
        n_jobs=2,
        random_state=42
    ))
])

model.fit(X_train, y_train)

predictions = model.predict(X_valid)

print("Validation records evaluated:", len(y_valid))
print("Validation accuracy:", accuracy_score(y_valid, predictions))
print("Validation macro F1:", f1_score(
    y_valid, predictions, average="macro", zero_division=0
))